# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.0.6


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    out = source.copy(deep=True)
    known_key = out["source_row"].notna()
    repeated = out.loc[known_key].drop_duplicates()
    assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()
    out["county_code"] = out["county_code"].str.strip()
    parse_counts = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        parse_counts[field] = int((out[field].notna() & numbers.isna()).sum())
        out[field] = numbers
    reason = pd.Series("", index=out.index, dtype="string")
    reason.loc[out["source_row"].isna()] += "unknown_source_row;"
    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"
    reason.loc[~out["action_taken"].isin([1, 2, 3])] += "invalid_action_taken;"
    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()
    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]
    audit = {
        "input": len(source), "replays": replay_n,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": parse_counts,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

## H2. Год и известный доход

In [4]:
clean, quarantine, audit = clean_export(export)
year_ok = clean["activity_year"].eq("2023").fillna(False).all()
print("Все activity_year == '2023':", year_ok)

print(audit)

n_total = len(clean)
income_known = clean["income"].notna().sum()
income_unknown = clean["income"].isna().sum()
mean_income = clean["income"].mean()

print("Записей в clean:", n_total)
print("Известный income:", income_known)
print("Неизвестный income:", income_unknown)
print("Средний известный доход:", round(mean_income, 2), "тыс. USD в год")

assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]

assert_frame_equal(export, raw_copy)

clean.head(3)

Все activity_year == '2023': True
{'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'loan_amount': 1, 'income': 1, 'action_taken': 0}}
Записей в clean: 6828
Известный income: 6485
Неизвестный income: 343
Средний известный доход: 237.07 тыс. USD в год


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,<NA>
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H3. Проверка отчётного года

In [5]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("year_ok для wrong_year:", year_ok)
assert not year_ok, "Проверка не обнаружила нарушенный год"
print("Нарушение обнаружено")

year_ok_clean = clean["activity_year"].eq("2023").fillna(False).all()
print("year_ok для clean:", year_ok_clean)

assert_frame_equal(export, raw_copy)

year_ok для wrong_year: False
Нарушение обнаружено
year_ok для clean: True


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей